# 无 ROS 指令客户端
先在驱动环境运行 `python -m rsim.apps.chassis_service --simulate --enable-motion`，或者使用硬件配置启动服务。此 Notebook 可在独立 Python 环境运行；只连接服务，不启动硬件。

默认只调用 `move(0)` / `rotate(0)`。非零示例应连接模拟服务后再开启。

In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file() and (p / 'rsim').is_dir())
os.environ['NOTEBOOK_ASSETS_ROOT'] = str(ROOT / 'assets')
from rsim.devices import Chassis
from rsim.runtime import Runtime
from rsim.transport.descriptor import TransportConfig
NAME = os.environ.get('RSIM_CHASSIS_NAME', 'chassis')
DOMAIN = int(os.environ.get('RSIM_CONTROL_DOMAIN', '0'))
RUN_SIMULATED_MOTION = False


In [ ]:
chassis = Chassis(NAME, transport=TransportConfig(domain_id=DOMAIN))
async with Runtime(chassis):
    frame = await chassis.pose.get(timeout=5)
    print(frame.data)
    historical = await chassis.pose.get(timestamp_ns=frame.stamp_ns, clock=frame.clock)
    assert historical is frame
    print(await chassis.move(0))
    print(await chassis.rotate(yaw_deg=0))
    if RUN_SIMULATED_MOTION:
        await chassis.move(.1)
        await chassis.move(-.1)
        await chassis.rotate(yaw_deg=20)
        await chassis.rotate(yaw_rad=-0.3490658504)


取消正在 await 的移动任务会请求停车；关闭 Runtime 会释放控制会话。服务端还独立检查租约期限。多个客户端可读位姿，同一时刻只接受一个控制所有者。